In [4]:
import numpy as np
import openvino as ov
from openvino import opset13 as opset
from openvino._pyopenvino.properties.hint import inference_precision

import nncf

# def get_one_layer_model():
#     input_node = opset.parameter([2, 3, 10], name="Input_1")
#     weights_data = np.random.randn(10, 10)
#     weights_data[-1, -1] = 10000
#     current_weights = opset.constant(weights_data, dtype=np.float32, name="weights")
#     matmul_node = opset.matmul(input_node, current_weights, transpose_a=False, transpose_b=True, name="MatMul")

#     weights_data_2 = np.random.randn(4, 10)
#     current_weights_2 = opset.constant(weights_data_2, dtype=np.float32, name="weights")
#     matmul_node_2 = opset.matmul(matmul_node, current_weights_2, transpose_a=False, transpose_b=True, name="MatMul")

#     result = opset.result(matmul_node_2, name="Result")
#     result.get_output_tensor(0).set_names(set(["Result"]))
#     model = ov.Model([result], [input_node])
#     return model


# model = get_one_layer_model()
# ov.save_model(model, "original.xml")

# model = ov.Core().read_model("original.xml")
model = ov.Core().read_model("e2m1_compressed.xml")
# model = nncf.compress_weights(
#     model,
#     # mode=nncf.CompressWeightsMode.E2M1,
#     mode=nncf.CompressWeightsMode.INT4_SYM,
#     group_size=2,
#     all_layers=True,
# )

# # ov.save_model(model, "e2m1_compressed.xml")
# ov.save_model(model, "int4_e2m1_compressed.xml")


# if not type_list:
#     return all_nodes_of_type
# for nncf_node in self.nodes.values():
#     if nncf_node.node_type in type_list:
#         all_nodes_of_type.append(nncf_node)
# return all_nodes_of_type

In [ ]:
model = nncf.compress_weights(
    model,
    mode=nncf.CompressWeightsMode.E2M1,
    group_size=2,
    all_layers=True,
)

In [10]:
print(*model.get_ops(), sep="\n")

<Parameter: 'Input_1' ([2,3,10], float)>
<Result: 'Result' ([2,3,4])>
<MatMul: 'MatMul' ([2,3,4])>
<Convert: 'weights' ([4,10])>
<Reshape: 'Reshape_56' ([4,10])>
<Constant: 'Constant_55' ([2])>
<Multiply: 'weights_compressed/fq_weights_1' ([4,5,2])>
<Convert: 'Convert_53' ([4,5,1])>
<Constant: 'weights_compressed/scale' ([4,5,1])>
<Convert: 'Convert_51' ([4,5,2])>
<Constant: 'weights_compressed' ([4,5,2])>
<MatMul: 'MatMul0' ([2,3,10])>
<Convert: 'weights0' ([10,10])>
<Reshape: 'Reshape_49' ([10,10])>
<Constant: 'Constant_48' ([2])>
<Multiply: 'weights_compressed0/fq_weights_1' ([10,5,2])>
<Convert: 'Convert_46' ([10,5,1])>
<Constant: 'weights_compressed0/scale' ([10,5,1])>
<Convert: 'Convert_44' ([10,5,2])>
<Constant: 'weights_compressed0' ([10,5,2])>


In [20]:
def get_ov_model(weight_shape, scale_shape, orig_shape):
    from nncf.openvino.optimized_functions.models import OVModelParameters
    from nncf.tensor.definitions import TensorDataType
    from nncf.tensor.functions.openvino_numeric import DTYPE_MAP as DTYPE_MAP_OV

    compressed_weight_dtype = TensorDataType.f4e2m1
    scale_dtype = TensorDataType.f8e8m0
    default_input_dtypes = {
        "scale": scale_dtype,
        "compressed_weight": compressed_weight_dtype,
    }

    default_output_dtypes = {
        "decompressed_weight": TensorDataType.float16,
    }

    # Update input and output dtypes with the default values
    ov_model_params = OVModelParameters()
    ov_model_params.input_dtypes = {**default_input_dtypes, **ov_model_params.input_dtypes}
    ov_model_params.output_dtypes = {**default_output_dtypes, **ov_model_params.output_dtypes}

    compressed_weight_op = opset.parameter(weight_shape, name="weight", dtype=DTYPE_MAP_OV[compressed_weight_dtype])
    scale_op = opset.parameter(scale_shape, name="scale", dtype=DTYPE_MAP_OV[scale_dtype])
    ov_parameters = [compressed_weight_op, scale_op]
    convert_w_op = opset.convert(compressed_weight_op, ov.Type.f16)
    convert_s_op = opset.convert(scale_op, ov.Type.f16)
    multiply_op = opset.multiply(convert_w_op, convert_s_op)
    reshape_op = opset.reshape(multiply_op, output_shape=orig_shape, special_zero=False)
    ov_results = [reshape_op]
    model = ov.Model(ov_results, ov_parameters)
    return ov.compile_model(model, device_name="CPU", config={inference_precision(): ov.Type.f32})


In [ ]:
inputs = np.ones([2, 30, 10000])

In [ ]:
compiled_model_orig = ov.compile_model(model, device_name="CPU", config={inference_precision(): ov.Type.f32})

In [ ]:
compiled_model_orig(inputs)

In [ ]:
res_orig = compiled_model_orig(inputs)

In [ ]:
[re_op] = [op for op in model.get_ops() if op.friendly_name == "Reshape_33"]

In [ ]:
[w_op] = [op for op in model.get_ops() if op.friendly_name == "weights_compressed0"]

In [ ]:
e2m1_tensor = ov.Tensor(w_op.data, w_op.get_output_shape(0), w_op.get_element_type())


In [29]:
e2m1_tensor

<Tensor: shape[10,5,2] type: f4e2m1>

In [8]:
[s_op] = [op for op in model.get_ops() if op.friendly_name == "weights_compressed0/scale"]

In [30]:
e8m0_tensor = ov.Tensor(s_op.data, s_op.get_output_shape(0), s_op.get_element_type())

In [21]:
ov_model = get_ov_model(w_op.shape, s_op.shape, [10,10])

In [22]:
ov_model

<CompiledModel:
inputs[
<ConstOutput: names[weight] shape[10,5,2] type: f4e2m1>,
<ConstOutput: names[scale] shape[10,5,1] type: f8e8m0>
]
outputs[
<ConstOutput: names[Result_13327] shape[10,10] type: f16>
]>

In [32]:
infer_request = ov_model.create_infer_request()

In [33]:
outputs = infer_request.infer(
    [e2m1_tensor, e8m0_tensor], share_inputs=True, share_outputs=False
)

In [34]:
outputs

{<ConstOutput: names[Result_13327] shape[10,10] type: f16>: array([[-1.500e+00,  2.500e-01, -1.000e+00, -2.000e+00,  2.000e+00,
         2.000e+00,  5.000e-01, -3.750e-01, -7.500e-01, -1.000e+00],
       [ 9.375e-02, -4.688e-02, -3.000e+00, -5.000e-01, -1.000e+00,
        -5.000e-01,  3.125e-02, -3.750e-01, -1.500e+00, -2.000e+00],
       [ 1.500e+00, -2.000e+00, -5.000e-01,  3.750e-01,  1.000e+00,
         1.500e+00,  6.250e-02,  3.750e-01, -1.500e+00,  3.750e-01],
       [ 5.000e-01, -1.250e-01,  0.000e+00,  1.000e+00,  7.500e-01,
         2.500e-01, -3.750e-01, -7.500e-01,  9.375e-02, -2.500e-01],
       [ 3.750e-01,  5.000e-01, -3.750e-01, -7.500e-01, -2.500e-01,
        -1.500e+00,  7.500e-01,  1.000e+00, -7.500e-01, -7.500e-01],
       [-1.000e+00, -7.500e-01, -1.500e+00, -1.500e+00,  1.000e+00,
         1.500e+00,  1.500e+00, -2.500e-01, -2.500e-01,  5.000e-01],
       [-2.500e-01,  5.000e-01, -1.250e-01,  1.000e+00, -1.500e+00,
        -0.000e+00,  3.000e+00,  1.000e+00, -7.500

In [ ]:
# w_param = opset.parameter(
#     shape=w_op.shape,
#     dtype=w_op.get_element_type(),
#     name=w_op.friendly_name,
# )
# for consumer in w_op.output(0).get_target_inputs():
#     consumer.replace_source_output(w_param.output(0))
# s_param = opset.parameter(
#     shape=s_op.shape,
#     dtype=s_op.get_element_type(),
#     name=s_op.friendly_name,
# )
# for consumer in s_op.output(0).get_target_inputs():
#     consumer.replace_source_output(s_param.output(0))

result = opset.result(re_op, name="Result")
result.get_output_tensor(0).set_names(set(["Result"]))
subgraph_model = ov.Model([result], [])
# ov.save_model(subgraph_model, "subgraph.xml")


In [ ]:
w_op.shape

In [ ]:
compiled_model = ov.compile_model(subgraph_model, device_name="CPU")  # , config={inference_precision(): ov.Type.f32})

In [ ]:
inputs = ([w_op.data, w_op.data],)

In [ ]:
res = compiled_model()

In [ ]:
res2 = compiled_model()

In [ ]:
del res2